# Notebook 2 — Oracle Retrain on Retain Set

**Experiment:** CMF-based machine unlearning benchmark — Oracle Retrain baseline.

**Purpose:** Train a fresh model from scratch on the **RETAIN SET ONLY** per seed (seeds 0,1,2), then evaluate with all 3 paper metrics.

**Prerequisite:** Run **Notebook 1** first to generate the dataset split files and configuration.

In [ ]:
import subprocess, sys
def sh(cmd, verbose=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if verbose and r.stdout: print(r.stdout[-4000:])
    if r.returncode != 0 and r.stderr: print('STDERR:', r.stderr[-2000:])
    return r.returncode
sh('pip install -q timm einops scikit-learn matplotlib seaborn')

In [ ]:
import os, sys, json, random, argparse, collections, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib; import matplotlib.pyplot as plt
matplotlib.rcParams.update({'figure.dpi': 110})
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU    :', torch.cuda.get_device_name(0))

In [ ]:
REPO_DIR = '/kaggle/working/CMF_Unlearning'
if not os.path.isdir(REPO_DIR):
    sh(f'git clone https://github.com/tiensinh2/CMF_Unlearning.git {REPO_DIR}')
else:
    sh(f'git -C {REPO_DIR} pull origin main')
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
CKPT_DATASET_DIR = '/kaggle/input/regun-notebook1'

_CONFIG_CANDIDATES = [
    f'{CKPT_DATASET_DIR}/regun_config.json',
    f'{CKPT_DATASET_DIR}/checkpoints/regun/regun_config.json',
    f'{CKPT_DATASET_DIR}/regun/regun_config.json',
]

config_path = None
CKPT_ROOT_NB1 = None
for _p in _CONFIG_CANDIDATES:
    if os.path.exists(_p):
        config_path = _p
        CKPT_ROOT_NB1 = os.path.dirname(_p)
        break

assert config_path is not None, (
    'regun_config.json not found. Checked:\n' +
    '\n'.join(f'  - {p}' for p in _CONFIG_CANDIDATES) + '\n' +
    'Make sure Notebook 1 finished and the correct dataset is attached.')

with open(config_path) as f:
    CFG = json.load(f)

TEST_MODE         = CFG['TEST_MODE']
TEST_FRACTION     = CFG.get('TEST_FRACTION', 0.01)
_MODE_TAG         = CFG.get('_MODE_TAG', 'test' if TEST_MODE else 'full')
DATASET           = CFG['DATASET']
ARCH              = CFG['ARCH']
IS_VIT            = CFG['IS_VIT']
NUM_CLASSES       = CFG['NUM_CLASSES']
CLASS_LABEL_NAMES = CFG['CLASS_LABEL_NAMES']
SPLIT_SEEDS       = CFG['SPLIT_SEEDS']
FORGET_FRACTION   = CFG['FORGET_FRACTION']
PRETRAIN_LR       = CFG['PRETRAIN_LR']
PRETRAIN_EPOCHS   = CFG['PRETRAIN_EPOCHS']
PRETRAIN_BS       = CFG['PRETRAIN_BS']
PRETRAIN_PATIENCE = CFG['PRETRAIN_PATIENCE']

_TOTAL     = {DATASET: CFG['TOTAL']}
_PER_CLASS = {DATASET: CFG['PER_CLASS']}

# Re-root checkpoint paths
_old_root = CFG['CKPT_ROOT']
def _repath(p): return p.replace(_old_root, CKPT_ROOT_NB1)
CKPT_PRETRAIN = _repath(CFG['CKPT_PRETRAIN'])

# Splits directory
SPLIT_DIR = _repath(CFG['SPLIT_DIR'])

for p, name in [(CKPT_PRETRAIN, 'pre_train')]:
    print(f'  [{"OK" if os.path.exists(p) else "MISSING"}] {name}: {p}')

DATA_PATH = '/kaggle/working/data'
CKPT_ROOT_NB2 = '/kaggle/working/checkpoints/regun_nb2'
os.makedirs(f'{CKPT_ROOT_NB2}/retrain', exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)

print(f'\nMode={"TEST" if TEST_MODE else "FULL"}  Dataset={DATASET}  Arch={ARCH}')
print(f'Split seeds: {SPLIT_SEEDS}  Forget fraction: {FORGET_FRACTION}')

In [ ]:
from utils import get_dataset, get_model, test, SubSet
import utils as _utils_module, functools

_orig_test = test
@functools.wraps(_orig_test)
def test(*a, verbose=False, **kw): return _orig_test(*a, verbose=verbose, **kw)
_utils_module.test = test

def make_args(**ov):
    d = dict(
        dataset=DATASET, arch=ARCH, data_path=DATA_PATH,
        num_classes=NUM_CLASSES, class_label_names=CLASS_LABEL_NAMES,
        batch_size=PRETRAIN_BS, test_batch_size=256,
        epochs_or_steps=PRETRAIN_EPOCHS,
        lr=PRETRAIN_LR, momentum=0.9, weight_decay=5e-4, gamma=0.5,
        seed=42, log_interval=200, val_ratio=0.1,
        patience=PRETRAIN_PATIENCE, warmup_epochs=5, min_lr=1e-5, lr_scheduler='cosine',
        unlearn_method='pre_train', unlearn_class=[],
        num_retain_samples=_TOTAL[DATASET], num_forget_samples=0,
        grad_norm_clip=None, salun_threshold=0.5,
        goel_exact=False, ssd_lambda=1, ssd_alpha=10,
        scrub_del_bsz=64, scrub_sgda_bsz=64, scrub_msteps=2, scrub_epochs=3,
        SVD_alpha_r=100, SVD_alpha_f=3, SVD_samples=900, SVD_max_patches=10000,
        tarun_impair_lr=2e-4, tarun_samples_per_class=1000,
        no_cuda=False, no_mps=True, dry_run=False,
        save_model=True, save_path=None,
        sub_set_mode=False, sub_set_samples=10000,
        no_train_transform=False, train_transform=True,
        gpu_id=0, multiclass=False, class_names=None,
        do_mia=False, do_mia_ulira=False, plot_mia_roc=False,
        prob_batch_size=128,
        freeze_except_last=False, zero_last_layer=False,
        remove_FC=True, CMF_momentum=0.9, CMFClassifier=True,
        do_lp=False, lp_every=0, ncc_every=0,
        eval_every_iter=0, lp_every_iter=0, ncc_every_iter=0,
        pretrained=IS_VIT,
        project_name='kaggle', group_name='regun',
    )
    d.update(ov)
    return argparse.Namespace(**d)

print('Helpers loaded.')

In [ ]:
args_base = make_args()
dataset_train, dataset_test = get_dataset(args_base)
print(f'Full dataset — Train={len(dataset_train)}  Test={len(dataset_test)}  Classes={NUM_CLASSES}')

if TEST_MODE:
    def _stratified_subset(ds, frac, seed=42):
        labels = (ds.targets if hasattr(ds, 'targets')
                  else [ds.dataset.targets[i] for i in ds.indices]
                  if hasattr(ds, 'indices') else [s[1] for s in ds.samples])
        rng = random.Random(seed)
        by_cls = collections.defaultdict(list)
        for i, l in enumerate(labels): by_cls[int(l)].append(i)
        kept = []
        for c in sorted(by_cls):
            pool = by_cls[c]; rng.shuffle(pool)
            kept.extend(pool[:max(1, math.ceil(len(pool)*frac))])
        sub = torch.utils.data.Subset(ds, kept)
        base_t = ds.targets if hasattr(ds, 'targets') else [ds.dataset.targets[i] for i in ds.indices]
        sub.targets = [base_t[i] for i in kept]
        return sub
    dataset_train = _stratified_subset(dataset_train, TEST_FRACTION)
    dataset_test  = _stratified_subset(dataset_test,  TEST_FRACTION)
    print(f'TEST_MODE: Train={len(dataset_train)}  Test={len(dataset_test)}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

TEST_KW = dict(batch_size=min(256, len(dataset_test)), num_workers=2, pin_memory=True, shuffle=False)
test_loader = torch.utils.data.DataLoader(dataset_test, **TEST_KW)

# Load splits
splits = {}
for seed in SPLIT_SEEDS:
    split_file = f'{SPLIT_DIR}/forget_indices_seed{seed}.json'
    if not os.path.exists(split_file):
        raise FileNotFoundError(f'Split file missing: {split_file}\nMake sure Notebook 1 was run and the dataset is attached correctly.')
    with open(split_file) as f:
        splits[seed] = json.load(f)
    n_f = splits[seed]['n_forget']
    n_r = splits[seed]['n_retain']
    print(f'Seed {seed}: forget={n_f}  retain={n_r}  forget%={100*n_f/(n_f+n_r):.1f}%')

print(f'All {len(SPLIT_SEEDS)} splits loaded.')

In [ ]:
# ── Test-set retain/forget split helper ──────────────────────────────
def _test_split_30_70(yte_numpy):
    """Split test indices into retain (70%) / forget (30%) per class, seed=0.
    Identical split used by probe and NCC so comparisons are apples-to-apples."""
    rng = random.Random(0)
    by_cls = collections.defaultdict(list)
    for i, l in enumerate(yte_numpy): by_cls[int(l)].append(i)
    fgt, ret = [], []
    for c in sorted(by_cls):
        pool = list(by_cls[c]); rng.shuffle(pool)
        nf = max(1, round(len(pool) * 0.30))
        fgt.extend(pool[:nf]); ret.extend(pool[nf:])
    return ret, fgt


# ── Metric 1: Output accuracy over arbitrary index set ────────────────
def eval_output_on_indices(model, dataset, indices, device, batch_size=256):
    """Standard forward-pass accuracy over a sample index set."""
    if len(indices) == 0: return float('nan')
    loader = torch.utils.data.DataLoader(
        SubSet(dataset, indices), batch_size=batch_size,
        shuffle=False, num_workers=2, pin_memory=True)
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total   += y.size(0)
    return correct / max(1, total)


# ── Metric 2: Linear Probe (paper section 3.2) ────────────────────────
@torch.no_grad()
def _extract_features(model, loader, device):
    """Extract penultimate features using the model's extract_features() method."""
    model.eval(); Xs, ys = [], []
    for x, y in loader:
        x = x.to(device)
        # Use extract_features if available (CMF models), else penultimate hook
        if hasattr(model, 'extract_features'):
            f = model.extract_features(x)
        else:
            f = model(x)  # fallback for non-CMF models with FC removed
        Xs.append(f.cpu()); ys.append(y)
    return torch.cat(Xs, 0).float(), torch.cat(ys, 0).long()


def eval_probe_on_indices(model, dataset_train, dataset_test,
                          retain_indices, forget_indices,
                          device, num_classes,
                          probe_epochs=100, probe_lr=0.01, batch_size=256):
    """
    Linear Probe accuracy — paper Section 3.2.
    Train a FRESH linear classifier on frozen encoder features from FULL training
    set (D_r union D_f), then evaluate on test set using _test_split_30_70.
    Returns (probe_retain_acc, probe_forget_acc).
    """
    model.eval()
    for p in model.parameters(): p.requires_grad_(False)

    full_train_loader = torch.utils.data.DataLoader(
        dataset_train, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True)
    Xtr, ytr = _extract_features(model, full_train_loader, device)

    test_loader_probe = torch.utils.data.DataLoader(
        dataset_test, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True)
    Xte, yte = _extract_features(model, test_loader_probe, device)

    head = nn.Linear(Xtr.size(1), num_classes).to(device)
    opt  = optim.SGD(head.parameters(), lr=probe_lr, momentum=0.9, weight_decay=0.0)
    ldr_probe = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for _ in range(probe_epochs):
        head.train()
        for bx, by in ldr_probe:
            opt.zero_grad()
            F.cross_entropy(head(bx.to(device)), by.to(device)).backward()
            opt.step()

    head.eval()
    with torch.no_grad():
        pred_te = head(Xte.to(device)).argmax(1).cpu()

    retain_test_indices, forget_test_indices = _test_split_30_70(yte.numpy())

    def _idx_acc(idxs):
        if not idxs: return float('nan')
        return float((pred_te[idxs] == yte[idxs]).float().mean())

    for p in model.parameters(): p.requires_grad_(True)
    return _idx_acc(retain_test_indices), _idx_acc(forget_test_indices)


# ── Metric 3: NCC — Nearest Class Center (paper eq. 5) ───────────────
def eval_ncc_on_indices(model, dataset_train, dataset_test,
                        retain_indices, forget_indices,
                        device, num_classes, batch_size=256):
    """
    NCC accuracy — paper Section 2.1 eq. (5).
    Classify each test sample by nearest class mean (cosine similarity) in
    L2-normalized feature space. Class means from full training set.
    Returns (ncc_retain_acc, ncc_forget_acc).
    """
    model.eval()
    full_loader = torch.utils.data.DataLoader(
        dataset_train, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True)
    Xtr, ytr = _extract_features(model, full_loader, device)
    Xtr_n = F.normalize(Xtr, dim=1)  # L2-normalize (consistent with CMF feature space)

    class_means = torch.zeros(num_classes, Xtr.size(1))
    for c in range(num_classes):
        mask = (ytr == c)
        if mask.any(): class_means[c] = Xtr_n[mask].mean(0)
    class_means_n = F.normalize(class_means, dim=1)  # [K, D]

    test_loader_ncc = torch.utils.data.DataLoader(
        dataset_test, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True)
    Xte, yte = _extract_features(model, test_loader_ncc, device)
    Xte_n = F.normalize(Xte, dim=1)

    with torch.no_grad():
        pred = (Xte_n @ class_means_n.t()).argmax(1)  # [N_test, K] -> argmax

    retain_test_idx, forget_test_idx = _test_split_30_70(yte.numpy())

    def _idx_acc(idxs):
        if not idxs: return float('nan')
        return float((pred[idxs] == yte[idxs]).float().mean())

    return _idx_acc(retain_test_idx), _idx_acc(forget_test_idx)


# ── Combined: all 3 metrics ───────────────────────────────────────────
def eval_three_metrics(model, dataset_train, dataset_test,
                       retain_indices, forget_indices,
                       device, num_classes,
                       run_probe=True, run_ncc=True,
                       probe_epochs=100):
    """
    Compute all 3 paper metrics (Output, Linear Probe, NCC) over index sets.
    Returns a flat dict with all 6 accuracy values.
    """
    # Output
    out_r = eval_output_on_indices(model, dataset_train, retain_indices, device)
    out_f = eval_output_on_indices(model, dataset_train, forget_indices,  device)

    # Linear Probe
    if run_probe and not TEST_MODE:
        pr_r, pr_f = eval_probe_on_indices(
            model, dataset_train, dataset_test,
            retain_indices, forget_indices, device, num_classes,
            probe_epochs=probe_epochs)
    else:
        pr_r = pr_f = float('nan')

    # NCC
    if run_ncc and not TEST_MODE:
        ncc_r, ncc_f = eval_ncc_on_indices(
            model, dataset_train, dataset_test,
            retain_indices, forget_indices, device, num_classes)
    else:
        ncc_r = ncc_f = float('nan')

    return dict(
        output_retain_acc=out_r, output_forget_acc=out_f,
        probe_retain_acc=pr_r,   probe_forget_acc=pr_f,
        ncc_retain_acc=ncc_r,    ncc_forget_acc=ncc_f,
    )


print('Three-metric eval harness defined (Output / Linear Probe / NCC).')


## C. Oracle Retrain per Seed

Train a fresh model from scratch on the **RETAIN SET ONLY** per seed (seeds 0,1,2), then evaluate with all 3 paper metrics.

In [ ]:
RETRAIN_RESULTS = []
LOADER_KW = dict(batch_size=PRETRAIN_BS, num_workers=2, pin_memory=True, shuffle=True, drop_last=True)

for seed in SPLIT_SEEDS:
    split = splits[seed]
    retain_indices = split['retain_indices']
    forget_indices = split['forget_indices']

    print(f'\n{"="*60}')
    print(f'  SEED {seed} — Retrain on retain set ({len(retain_indices)} samples)')
    print(f'{"="*60}')

    ckpt_out = f'{CKPT_ROOT_NB2}/retrain/seed{seed}.pt'
    t0 = time.time()
    
    if os.path.exists(ckpt_out):
        print(f'  Checkpoint exists: {ckpt_out} — skipping training.')
        wall_clock_minutes = 0.0
    else: 
        # Fresh model from scratch
        args_rt = make_args(
            unlearn_method='retrain',
            epochs_or_steps=PRETRAIN_EPOCHS,
            lr=PRETRAIN_LR,
            patience=PRETRAIN_PATIENCE,
            num_classes=NUM_CLASSES,
            class_label_names=CLASS_LABEL_NAMES,
            remove_FC=True,
            CMFClassifier=True,
        )
        model_rt = get_model(args_rt, device)

        retain_ds  = SubSet(dataset_train, retain_indices)
        retain_loader = torch.utils.data.DataLoader(retain_ds, **LOADER_KW)

        optimizer = optim.SGD(
            model_rt.parameters(), lr=PRETRAIN_LR,
            momentum=0.9, weight_decay=5e-4, nesterov=True
        )
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=PRETRAIN_EPOCHS
        )

        for epoch in range(1, PRETRAIN_EPOCHS + 1):
            model_rt.train()
            total_loss = total_n = 0
            if hasattr(model_rt, 'recompute_cmf'):
                model_rt.eval()
                model_rt.recompute_cmf(retain_loader, device=device)
                model_rt.train()
            
            for x, y in retain_loader:
                x, y = x.to(device), y.to(device)
                optimizer.zero_grad()
                logits = model_rt(x)
                loss = F.cross_entropy(logits, y)
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * x.size(0)
                total_n += x.size(0)
            scheduler.step()
            
            if epoch % max(1, PRETRAIN_EPOCHS // 10) == 0 or epoch == PRETRAIN_EPOCHS:
                elapsed = time.time() - t0
                print(f'  Epoch {epoch:3d}/{PRETRAIN_EPOCHS}  ' + 
                      f'loss={total_loss/max(1,total_n):.4f}  ' + 
                      f'elapsed={elapsed:.0f}s')

        if hasattr(model_rt, 'recompute_cmf'):
            model_rt.eval()
            model_rt.recompute_cmf(retain_loader, device=device)

        torch.save(model_rt.state_dict(), ckpt_out)
        t1 = time.time()
        wall_clock_minutes = (t1 - t0) / 60.0
        print(f'  → saved: {ckpt_out}  (wall time: {wall_clock_minutes:.1f} min)')

    # Reload checkpoint and evaluate using the 3-metric harness
    args_ev = make_args(
        unlearn_method='retrain',
        num_classes=NUM_CLASSES,
        class_label_names=CLASS_LABEL_NAMES,
        remove_FC=True,
        CMFClassifier=True,
    )
    model_ev = get_model(args_ev, device)
    model_ev.load_state_dict(torch.load(ckpt_out, map_location=device))

    # CMF recompute one final time
    retain_ds = SubSet(dataset_train, retain_indices)
    retain_loader_ev = torch.utils.data.DataLoader(retain_ds, batch_size=PRETRAIN_BS, shuffle=False, num_workers=2, pin_memory=True)
    if hasattr(model_ev, 'recompute_cmf'):
        model_ev.eval()
        model_ev.recompute_cmf(retain_loader_ev, device=device)

    print(f'Evaluating Seed {seed} Retrained model...')
    metrics = eval_three_metrics(
        model_ev, dataset_train, dataset_test,
        retain_indices, forget_indices, device, NUM_CLASSES
    )

    res = dict(
        seed=seed,
        n_retain=len(retain_indices),
        n_forget=len(forget_indices),
        output_retain_acc=metrics['output_retain_acc'],
        output_forget_acc=metrics['output_forget_acc'],
        probe_retain_acc=metrics['probe_retain_acc'],
        probe_forget_acc=metrics['probe_forget_acc'],
        ncc_retain_acc=metrics['ncc_retain_acc'],
        ncc_forget_acc=metrics['ncc_forget_acc'],
        wall_clock_minutes=wall_clock_minutes if not os.path.exists(ckpt_out) or 'wall_clock_minutes' not in locals() else 0.0,
        ckpt_path=ckpt_out
    )
    if 'wall_clock_minutes' not in locals():
        res['wall_clock_minutes'] = 0.0
    
    print(f'Seed {seed} Retrained results:')
    for k, v in res.items():
        if isinstance(v, float) and k != 'wall_clock_minutes':
            print(f'  {k}: {v*100:.2f}%')
        else:
            print(f'  {k}: {v}')

    RETRAIN_RESULTS.append(res)

print('\nAll oracle retrain runs complete.')

## D. Results Summary

Consolidate accuracy metrics across seeds, save results to CSV, and write downstream notebook config.

In [ ]:
df = pd.DataFrame(RETRAIN_RESULTS)
print('\n=== Oracle Retrain Results ===')
print(df.to_string(index=False))

cols = [
    'output_retain_acc', 'output_forget_acc',
    'probe_retain_acc', 'probe_forget_acc',
    'ncc_retain_acc', 'ncc_forget_acc'
]

print('\n=== Mean ± Std Table (matches paper Table 1 "Retain-only Retrain" row format) ===')
summary = {}
for col in cols:
    mean_val = df[col].mean()
    std_val = df[col].std()
    if pd.isna(std_val):
        std_val = 0.0
    summary[col] = f'{mean_val*100:.2f}% ± {std_val*100:.2f}%' if not (TEST_MODE and ('probe' in col or 'ncc' in col)) else 'NaN (TEST_MODE)'

summary_df = pd.DataFrame([summary])
print(summary_df.to_string(index=False))

# Save CSV
csv_path = f'/kaggle/working/results_oracle_retrain_{DATASET}_{ARCH}.csv'
df.to_csv(csv_path, index=False)
print(f'\nCSV saved to {csv_path}')

# Save nb2_config.json
nb2_config = {
    'CKPT_ROOT_NB2': CKPT_ROOT_NB2,
    'retrain_checkpoints': {str(r['seed']): r['ckpt_path'] for r in RETRAIN_RESULTS}
}
config_nb2_path = f'{CKPT_ROOT_NB2}/nb2_config.json'
with open(config_nb2_path, 'w') as f:
    json.dump(nb2_config, f, indent=2)
print(f'nb2_config.json saved to {config_nb2_path}')